# MinGRU Phase 1.3 — Colab A100 training

**Goal:** Reach coherent TinyStories generation on the A100 in a reasonable
wall-clock budget (target ≤ 1 hour for the Phase 1.3 gate).

**Architecture:** `MinGRUModel(d=256, L=6, d_ffn=768, vocab=4000)` — 5.75M
params, scaffolded in `cubemind/training/vsa_lm.py` (Phase 1.2) and
trained by `sandbox/mingru_baseline/train.py`.

**Optimizer:** `grilly.optim.AdamW`, cosine LR decay from 3e-4 → 1e-5,
warmup 1000, `grad_clip=1.0` (per `TASKS.md` Phase 1.3).

**Runtime:** `Runtime → Change runtime type → GPU (A100)`. The notebook
checks `grilly` imports after install — if grilly can't attach to the
Vulkan ICD on Colab's NVIDIA runtime, it falls back to its NumPy path
(slow but correct).

**Prerequisites:**
- cubemind + grilly repos installed on Colab (install cell below).
- `data/tinystories_50k.json` available. Upload the file once or let
  the notebook fall back to the HuggingFace V2 split download.

---

## 0 · Runtime check

In [ ]:
!nvidia-smi || echo 'no GPU attached — switch to GPU runtime'
import platform, sys
print('python :', sys.version.split()[0])
print('platform:', platform.platform())

## 1 · Install grilly + cubemind

Two ways to provide the repos:

- **Clone from GitHub** (default; requires the repos to be public or that
  you have a `GIT_TOKEN` secret set).
- **Upload zips** (fallback): drag-drop `grilly.zip` and `cubemind.zip` into
  Colab's Files pane and use the commented-out cell below.

`grilly` ships a compiled `grilly_core` Vulkan backend. On Colab's
Linux NVIDIA runtime that means the build must provide a Linux `.so` (if
available) or fall back to the numpy path inside grilly. Either works —
the training loop calls `grilly.nn.autograd` / `grilly.optim` which
handle both paths transparently.

In [ ]:
%cd /content
# --- Clone (edit branch/urls as needed) -----------------------------------
GRILLY_REPO  = 'https://github.com/Grillcheese-AI/grilly.git'
CUBEMIND_REPO = 'https://github.com/Grillcheese-AI/cubemind.git'
!rm -rf grilly cubemind
!git clone --depth 1 $GRILLY_REPO
!git clone --depth 1 $CUBEMIND_REPO
!ls

In [ ]:
# Pip installs.
# train_torch.py only needs torch/numpy/tokenizers — grilly isn't
# imported. Install grilly + cubemind anyway so the rest of the repo
# (tests, other phases) work on this Colab session.
#
# matplotlib-inline is pulled explicitly because Colab's shell env sets
# MPLBACKEND=module://matplotlib_inline.backend_inline and grilly's init
# chain would crash without it.
!pip install -q --upgrade pip setuptools wheel
!pip install -q numpy tokenizers torch loguru click dependency-injector fastapi pydantic matplotlib matplotlib-inline
!pip install -q --no-build-isolation ./grilly || echo 'grilly build failed (ok if using torch path)'
!pip install -q --no-build-isolation ./cubemind || echo 'cubemind install failed (ok if running train_torch.py directly)'
print('done')

In [ ]:
# Smoke-test imports. grilly prints GPU device info on first import; if
# Vulkan ICD isn't available, you'll see a numpy fallback note instead.
import grilly
from grilly import nn as gnn
from grilly.optim import AdamW
from grilly.nn.prefix_scan import prefix_scan_causal

print('grilly version  :', grilly.__version__ if hasattr(grilly, '__version__') else 'editable')
print('Vulkan available:', getattr(grilly, 'VULKAN_AVAILABLE', '?'))

## 2 · Verify the prefix-scan seq-len cap is lifted

Confirms the shader + binding rebuilt with the per-thread serial scan so
MinGRU can run at TinyStories seq=256+. If this fails with `seq_len > 32`,
the grilly install picked up an older release — reinstall from source.

In [ ]:
import numpy as np
from grilly.nn.prefix_scan import prefix_scan_causal

B, S, D = 2, 256, 64
rng = np.random.default_rng(0)
x = rng.standard_normal((B, S, D)).astype(np.float32) * 0.1
a = np.clip(0.05 + rng.random((B, S, D)).astype(np.float32) * 0.9, 0.05, 0.95)
h = prefix_scan_causal(x, a)

# Reference: numpy scalar scan
h_ref = np.zeros_like(x); hp = np.zeros((B, D), dtype=np.float32)
for t in range(S):
    hp = a[:, t] * hp + x[:, t]
    h_ref[:, t] = hp
print(f'seq=256 max abs err vs numpy: {float(np.abs(h.data - h_ref).max()):.3e}')

## 3 · Dataset — upload `tinystories_50k.json`

The training script prefers the local 50k-story JSON (same file used in
`cubemind/data/tinystories_50k.json`). Drop it into the Files pane at
`/content/cubemind/data/tinystories_50k.json`, OR uncomment the
HuggingFace fallback cell.

In [ ]:
import os
from pathlib import Path

target = Path('/content/cubemind/data/tinystories_50k.json')
target.parent.mkdir(parents=True, exist_ok=True)

if not target.exists():
    # Prompt for upload via Colab's file picker.
    try:
        from google.colab import files
        print('Choose tinystories_50k.json to upload (~45 MB):')
        uploaded = files.upload()
        for name, data in uploaded.items():
            if 'tinystories' in name.lower():
                target.write_bytes(data)
                break
    except Exception as e:
        print(f'Upload failed ({e}) — falling back to HuggingFace split in train.py')

if target.exists():
    print(f'dataset ready: {target} ({target.stat().st_size/1e6:.1f} MB)')
else:
    print('no local JSON — prepare_data() will auto-download TinyStoriesV2 GPT-4')

## 4 · Train — PyTorch backend on A100

Colab's A100 runtime has CUDA drivers but no Vulkan ICD, so grilly's
Vulkan backend silently falls back to numpy-on-CPU. The PyTorch mirror at
`sandbox/mingru_baseline/train_torch.py` implements the identical MinGRU
architecture (same math, same init, same loss) using `torch.nn` modules
and runs natively on CUDA.

Phase 1.5 re-validates the same architecture on the grilly stack once
Vulkan is sorted. For Phase 1.3 the coherence gate is architecture-level.

Defaults below match the A100 memory-optimization recommendations (large
batch, fewer steps, LR scaled by sqrt(batch)):

- `batch_size=128` (≈3 GB VRAM instead of 2)
- `steps=6000` at `lr=1e-3` gives the same ~200M-token budget as 20K×16
- `warmup=200` because the shorter run needs less ramp
- `eval_every=500` for 12 checkpoints of progress
- `dtype=bf16` — A100 autocast for ≈2× speedup without precision loss

In [ ]:
%cd /content/cubemind
import os
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ.pop('MPLBACKEND', None)

# Training knobs — A100 has ~80GB VRAM to burn.
STEPS       = 6_000
MINUTES_CAP = 90
LOG_EVERY   = 20
EVAL_EVERY  = 500
CKPT_EVERY  = 500
BATCH_SIZE  = 128
LR          = 1e-3
WARMUP      = 200
DTYPE       = 'bf16'         # fp32 / bf16 / fp16

!python -u sandbox/mingru_baseline/train_torch.py \
    --steps $STEPS --minutes $MINUTES_CAP \
    --batch-size $BATCH_SIZE --lr $LR --warmup $WARMUP \
    --log-every $LOG_EVERY --eval-every $EVAL_EVERY --ckpt-every $CKPT_EVERY \
    --dtype $DTYPE

## 4b · Restart — full HF V2 TinyStories corpus

The earlier run trained on only the first **10.9M tokens** of the
downloaded HF V2 corpus (that's the `--subset-tokens` default, sized for
the local 50k-story JSON). With ~500M+ tokens actually available, the
model overfit by step 3000 (train PPL 3.7 / val PPL 11.2 = 9 effective
epochs).

This cell:
1. Quarantines the local 50k JSON if present, so `prepare_data` commits
   to the HF source path.
2. Clears any stale `*_local50k_*` cache so token bins get rebuilt from
   the HF .txt files.
3. Clears `results_torch/` so the new best checkpoint isn't compared
   against a worse prior one.
4. Re-runs training with `--subset-tokens 200_000_000` — gives the
   6000-step × 32K-tok/step schedule ≈ 1 effective epoch.

The `data/train.txt` and `data/valid.txt` downloads from the last run
are kept — no 2 GB re-download.

In [ ]:
%cd /content/cubemind
import os, shutil
from pathlib import Path

# 1. Force HF source even if the 50k JSON was uploaded.
json_path = Path('/content/cubemind/data/tinystories_50k.json')
if json_path.exists():
    quarantine = json_path.with_suffix('.json.disabled')
    shutil.move(str(json_path), str(quarantine))
    print(f'  quarantined {json_path} -> {quarantine.name}')
else:
    print(f'  no local JSON at {json_path} (good, HF source will be used)')

# 2. Clear stale local50k caches if any previous run created them.
cache_dir = Path('/content/cubemind/sandbox/mingru_baseline/data')
for pattern in ('*local50k*',):
    for f in cache_dir.glob(pattern):
        f.unlink()
        print(f'  removed cache: {f.name}')

# 3. Keep HF .txt downloads (skip re-download), preview what's there.
for name in ('train.txt', 'valid.txt'):
    p = cache_dir / name
    if p.exists():
        print(f'  HF txt ready: {name} ({p.stat().st_size/1e6:.1f} MB)')

# 4. Reset results_torch/ so best.pt isn't stuck at an overfit step.
out = Path('/content/cubemind/sandbox/mingru_baseline/results_torch')
if out.exists():
    backup = out.with_name(f'results_torch_overfit_{int(__import__("time").time())}')
    shutil.move(str(out), str(backup))
    print(f'  archived prior run -> {backup.name}')
out.mkdir(parents=True, exist_ok=True)

# 5. Environment for unbuffered logging + MPLBACKEND defanged.
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ.pop('MPLBACKEND', None)

# 6. Train with a proper-size subset (~1 epoch on 6K steps at batch=128).
STEPS       = 6_000
MINUTES_CAP = 240
LOG_EVERY   = 20
EVAL_EVERY  = 500
CKPT_EVERY  = 500
BATCH_SIZE  = 128
LR          = 1e-3
WARMUP      = 200
DTYPE       = 'bf16'
SUBSET      = 200_000_000   # 200M tokens of HF V2 (~1 epoch over 6K steps)

!MPLBACKEND= python -u sandbox/mingru_baseline/train_torch.py \
    --steps $STEPS --minutes $MINUTES_CAP \
    --batch-size $BATCH_SIZE --lr $LR --warmup $WARMUP \
    --subset-tokens $SUBSET \
    --log-every $LOG_EVERY --eval-every $EVAL_EVERY --ckpt-every $CKPT_EVERY \
    --dtype $DTYPE

In [ ]:
# Toggle this if you want the (large) tokenized .bin files in Drive too.
# Off by default — they regenerate from train.txt + tokenizer.json in
# minutes if you actually need them on a future session.
COPY_BINS = False

# ── Mount Drive (no-op if already mounted) ──────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import shutil, time
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/cubemind/mingru_baseline')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

stamp = time.strftime('%Y%m%d_%H%M%S')
dest = DRIVE_BASE / f'run_{stamp}'
dest.mkdir(parents=True, exist_ok=True)
print(f'destination: {dest}\n')

# 1. Checkpoints + generations + summary
src_results = Path('/content/cubemind/sandbox/mingru_baseline/results_torch')
if src_results.exists():
    dest_results = dest / 'results_torch'
    shutil.copytree(src_results, dest_results, dirs_exist_ok=True)
    files = list(dest_results.rglob('*'))
    sz = sum(p.stat().st_size for p in files if p.is_file()) / 1e6
    print(f'  results_torch: {len([p for p in files if p.is_file()])} files, '
          f'{sz:.1f} MB')
else:
    print('  (no results_torch/ yet — train first)')

# 2. Tokenizer + meta (always small, always critical)
src_data = Path('/content/cubemind/sandbox/mingru_baseline/data')
dest_data = dest / 'data'
dest_data.mkdir(exist_ok=True)
tok_count = 0
for pattern in ('tokenizer_*.json', 'meta_*.json'):
    for f in sorted(src_data.glob(pattern)):
        shutil.copy2(f, dest_data / f.name)
        print(f'    {f.name} ({f.stat().st_size/1024:.1f} KB)')
        tok_count += 1
if tok_count == 0:
    print('  (no tokenizer cache yet)')

# 3. Tokenized bins (optional — large, regenerable)
if COPY_BINS:
    for f in sorted(src_data.glob('*.bin')):
        shutil.copy2(f, dest_data / f.name)
        print(f'    {f.name} ({f.stat().st_size/1e6:.1f} MB)')

# 4. Manifest so future-you knows what's in this dir
manifest = {
    'run_id': f'run_{stamp}',
    'saved_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'source_repo_head': None,
    'files': sorted(str(p.relative_to(dest)) for p in dest.rglob('*') if p.is_file()),
}
import subprocess
try:
    manifest['source_repo_head'] = subprocess.check_output(
        ['git', '-C', '/content/cubemind', 'rev-parse', 'HEAD'],
        text=True,
    ).strip()
except Exception as e:
    manifest['source_repo_head'] = f'(git unavailable: {e})'
import json
(dest / 'MANIFEST.json').write_text(json.dumps(manifest, indent=2))
print(f'\n  manifest: {dest / "MANIFEST.json"}')
print(f'\n  saved {len(manifest["files"]):,} files to Drive')
print(f'\n  to restore on a fresh Colab:')
print(f'      from google.colab import drive')
print(f"      drive.mount('/content/drive')")
print(f"      !cp -r '{dest}/results_torch' /content/cubemind/sandbox/mingru_baseline/")
print(f"      !cp -r '{dest}/data'/* /content/cubemind/sandbox/mingru_baseline/data/")

## 4.5 · Persist artefacts to Google Drive

Colab containers vanish on idle disconnect. This cell pushes the
checkpoints, generations, and — critically — **the exact tokenizer the
model was trained against** into Google Drive so eval doesn't break
with a BPE mismatch on the next session.

Run this **after each training run completes**, or before any cell that
might wipe `results_torch/` (the restart cell above moves the prior run
to a local backup dir, but Drive is the only persistent home).

What gets copied:
- `results_torch/` — `best.pt`, `checkpoint.pt`, `gen_step_*.md`,
  `generations_final.md`, `summary.json`
- `data/tokenizer_*.json` + `data/meta_*.json` — small (<1 MB), required
  for eval_coherence.py
- `data/*.bin` — tokenized uint16 corpus (skipped by default; large)

Each run lands in a timestamped subdir so multiple runs don't overwrite
each other. To restore on a fresh session, mount Drive and copy back.

## 5 · Inspect generations

`train_torch.py` writes `gen_step_<N>.md` every `--eval-every` steps plus
a `generations_final.md` at the end into `sandbox/mingru_baseline/results_torch/`.

In [ ]:
from pathlib import Path
results = Path('/content/cubemind/sandbox/mingru_baseline/results_torch')
gens = sorted(results.glob('gen_step_*.md'))
final = results / 'generations_final.md'
print('artefacts:')
for p in gens + ([final] if final.exists() else []):
    print(f'  {p.name}  ({p.stat().st_size/1024:.1f} KB)')
if final.exists():
    print('\n--- generations_final.md (first 4 KB) ---\n')
    print(final.read_text(encoding='utf-8')[:4000])
elif gens:
    latest = gens[-1]
    print(f'\n--- {latest.name} (first 4 KB) ---\n')
    print(latest.read_text(encoding='utf-8')[:4000])

## 6 · Download results

Zip up `results/` (checkpoints + generations + summary.json) and pull it
back to your machine. Commit to `sandbox/mingru_baseline/results/` in
the cubemind repo.

In [ ]:
import shutil
from pathlib import Path
results = Path('/content/cubemind/sandbox/mingru_baseline/results_torch')
assert results.exists(), 'nothing to download — training did not produce outputs'
out = shutil.make_archive('/content/mingru_baseline_results_torch', 'zip',
                          str(results))
print(f'packaged: {out}  ({Path(out).stat().st_size/1e6:.1f} MB)')

from google.colab import files
files.download(out)

## Post-run checklist

1. Extract `mingru_baseline_results_torch.zip` into
   `sandbox/mingru_baseline/results_torch/` in the local repo.
2. Review `generations_final.md` — visual coherence check per Phase 1.4.
3. If stories are grammatically correct and on-prompt, mark Phase 1.3
   done in `TASKS.md`. If not: more steps, adjust `seq_len`, or
   hyperparameters.
4. Phase 1.4 runs `eval_coherence.py` against `results_torch/best.pt`
   (same checkpoint format as grilly version) for GPT-4 rubric scores.
5. Phase 1.5 re-validates the identical architecture on grilly/Vulkan
   (RX 6750 XT locally, or Colab once Vulkan is enabled).